<a href="https://colab.research.google.com/github/Thangapandi1611/ml-safety-project/blob/main/week9/FGSM_RECALL_TRADEOFF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import recall_score
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader,Subset
import random
import pandas as pd
from PIL import Image
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Creating Custom Dataset Class
class CarlaDataset(Dataset):

    def __init__(self, data_path, label_column, transform=None):

        self.data_path = data_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(data_path, "labels.csv")
        )

        self.label_column = label_column

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        frame = str(row['frame']).zfill(6)

        img_path = os.path.join(
            self.data_path,
            "rgb-front",
            f"{frame}.jpg"
        )

        image = Image.open(img_path).convert("RGB")

        label = int(row[self.label_column])

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
#Image Transformation
#ResNet18 expects this input size, 224×224
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [4]:
#fGSM
def fgsm_attack(image, epsilon, data_grad):

    perturbed_image = image + epsilon * data_grad.sign()

    perturbed_image = torch.clamp(
        perturbed_image,
        0,
        1
    )

    return perturbed_image

In [32]:
#eV
import numpy as np
def evaluate_fgsm_recall(
    MODEL_PATH,
    DATASET_PATH,
    LABEL_COLUMN,
    epsilons=[0.01, 0.05, 0.1],
    sample_size=300
):

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "cpu"
    )

    # Load model
    model = models.resnet18(weights=None)

    model.fc = nn.Linear(
        model.fc.in_features,
        1
    )

    model.load_state_dict(
        torch.load(
            MODEL_PATH,
            map_location=device
        )
    )

    model.to(device)
    model.eval()

    criterion = nn.BCEWithLogitsLoss()

    # Dataset
    dataset = CarlaDataset(
        DATASET_PATH,
        label_column=LABEL_COLUMN,
        transform=transform
    )

    # Random 100 images
    indices = random.sample(
        range(len(dataset)),
        min(sample_size, len(dataset))
    )

    subset = Subset(dataset, indices)

    loader = DataLoader(
        subset,
        batch_size=1,
        shuffle=False
    )
    positive_count = 0

    for _, label in loader:
        positive_count += label.item()

    print("Positive samples:", positive_count)

    results = {}

    # ----------------------
    # CLEAN RECALL
    # ----------------------

    y_true = []
    y_pred = []

    with torch.no_grad():

        for image, label in loader:

            image = image.to(device)

            output = model(image)

            pred = (
                torch.sigmoid(output) >= 0.5
            ).int().cpu().item()

            y_true.append(label.item())
            y_pred.append(pred)

    clean_recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    results["Clean"] = clean_recall

    print(f"\nClean Recall: {clean_recall:.4f}")

    # ----------------------
    # FGSM RECALL
    # ----------------------

    for epsilon in epsilons:
        print(
              "Predictions:",
            np.unique(y_pred, return_counts=True)
            )
        y_true = []
        y_pred = []

        for image, label in loader:

            image = image.to(device)

            label_tensor = torch.tensor(
                [[label.item()]],
                dtype=torch.float32
            ).to(device)

            image.requires_grad = True

            output = model(image)

            loss = criterion(
                output,
                label_tensor
            )

            model.zero_grad()

            loss.backward()

            data_grad = image.grad.data

            adv_image = fgsm_attack(
                image,
                epsilon,
                data_grad
            )

            adv_output = model(
                adv_image
            )

            pred = (
                torch.sigmoid(adv_output) >= 0.5
            ).int().cpu().item()

            y_true.append(label.item())
            y_pred.append(pred)

        adv_recall = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

        results[epsilon] = adv_recall

        print(
            f"ε={epsilon:.2f} Recall: "
            f"{adv_recall:.4f}"
        )

    return results

In [34]:
#PEDESTRAIN MODEL
pedestrian_results = evaluate_fgsm_recall(
    MODEL_PATH="/content/drive/MyDrive/MLS/pedestrian_model.pth",
    DATASET_PATH="/content/drive/MyDrive/MLS/test",
    LABEL_COLUMN="has_pedestrian"
)

pedestrian_results

Positive samples: 56

Clean Recall: 0.2143
Predictions: (array([0, 1]), array([283,  17]))
ε=0.01 Recall: 0.0000
Predictions: (array([0, 1]), array([ 67, 233]))
ε=0.05 Recall: 0.2321
Predictions: (array([0, 1]), array([ 43, 257]))
ε=0.10 Recall: 0.7321


{'Clean': 0.21428571428571427,
 0.01: 0.0,
 0.05: 0.23214285714285715,
 0.1: 0.7321428571428571}

In [33]:
#VEHLICLE

VEHICLE_results = evaluate_fgsm_recall(
    MODEL_PATH="/content/drive/MyDrive/MLS/Vehicle_model.pth",
    DATASET_PATH="/content/drive/MyDrive/MLS/test",
    LABEL_COLUMN="has_vehicle"
)

VEHICLE_results

Positive samples: 226

Clean Recall: 0.8363
Predictions: (array([0, 1]), array([108, 192]))
ε=0.01 Recall: 0.3540
Predictions: (array([0, 1]), array([146, 154]))
ε=0.05 Recall: 1.0000
Predictions: (array([1]), array([300]))
ε=0.10 Recall: 1.0000


{'Clean': 0.8362831858407079, 0.01: 0.35398230088495575, 0.05: 1.0, 0.1: 1.0}

In [35]:
#TRAFFIC LIGHT MODEL

TL_results = evaluate_fgsm_recall(
    MODEL_PATH="/content/drive/MyDrive/MLS/Traffic_Light_model.pth",
    DATASET_PATH="/content/drive/MyDrive/MLS/test",
    LABEL_COLUMN="has_traffic_light"
)

TL_results

Positive samples: 223

Clean Recall: 0.9776
Predictions: (array([0, 1]), array([ 69, 231]))
ε=0.01 Recall: 0.3363
Predictions: (array([0, 1]), array([154, 146]))
ε=0.05 Recall: 0.0135
Predictions: (array([0, 1]), array([220,  80]))
ε=0.10 Recall: 0.0538


{'Clean': 0.9775784753363229,
 0.01: 0.336322869955157,
 0.05: 0.013452914798206279,
 0.1: 0.053811659192825115}

In [18]:
labels = pd.read_csv(
    "/content/drive/MyDrive/MLS/test/labels.csv"
)

In [19]:

labels['has_traffic_light'].value_counts()

,count
has_traffic_light,
True,2584
False,1016


In [20]:
labels['has_pedestrian'].value_counts()


,count
has_pedestrian,
False,2894
True,706


In [21]:
labels['has_vehicle'].value_counts()

,count
has_vehicle,
True,2700
False,900
